# 🏥 추가 과제: 범위·표준편차부터 조건부 확률까지

> 이 노트북은 어제 실습한 **[콜레스테롤 검진 데이터 - 관측치/변수/유형/비율/평균·중앙값/이상치 영향]**에 이어서,
> **범위 → 분산·표준편차 → 사분위수·IQR → 1일차 종합 → 분포 → 빈도 → 교차표 → 조건부 확률**까지 다룹니다.
>
> 모든 개념은 (1) 비전공자를 위한 이론 설명 → (2) 문제 → (3) 정답 순서로 구성되어 있습니다.
> 먼저 스스로 풀어본 뒤 정답을 확인하세요.

### 이 노트북에서 다루는 내용
| 순서 | 주제 | 1·2일차 대응 단원 |
|---|---|---|
| 1 | 범위(Range) | D1-06 |
| 2 | 분산과 표준편차 | D1-06 |
| 3 | 사분위수와 IQR | D1-07 |
| 4 | 1일차 종합 미니 리포트 | D1-08 |
| 5 | 분포의 이해 (대칭/치우침) | D2-01 |
| 6 | 빈도와 상대빈도 | D2-02 |
| 7 | 교차표 | D2-03 |
| 8 | 조건부 비율과 조건부 확률 | D2-04 |


## 📋 실습 데이터 다시 불러오기

어제 사용한 콜레스테롤 검진 환자 9명 데이터에, 오늘 실습에 필요한 **가족력(family_history)** 정보를 추가했습니다.
(가족력: 직계가족 중 고지혈증/당뇨병 진단을 받은 사람이 있으면 1, 없으면 0)


In [18]:
patients_new = [
    {"patient_id": "Q001", "age": 33, "sex": "F", "ldl_cholesterol": 95,  "smoking": 0, "family_history": 0},
    {"patient_id": "Q002", "age": 45, "sex": "M", "ldl_cholesterol": 110, "smoking": 1, "family_history": 1},
    {"patient_id": "Q003", "age": 51, "sex": "F", "ldl_cholesterol": 102, "smoking": 0, "family_history": 0},
    {"patient_id": "Q004", "age": 39, "sex": "M", "ldl_cholesterol": 88,  "smoking": 0, "family_history": 0},
    {"patient_id": "Q005", "age": 62, "sex": "F", "ldl_cholesterol": 130, "smoking": 1, "family_history": 1},
    {"patient_id": "Q006", "age": 58, "sex": "M", "ldl_cholesterol": 115, "smoking": 1, "family_history": 1},
    {"patient_id": "Q007", "age": 47, "sex": "F", "ldl_cholesterol": 99,  "smoking": 0, "family_history": 0},
    {"patient_id": "Q008", "age": 41, "sex": "M", "ldl_cholesterol": 105, "smoking": 0, "family_history": 1},
    {"patient_id": "Q009", "age": 69, "sex": "M", "ldl_cholesterol": 245, "smoking": 1, "family_history": 1},
]

import statistics as stats

print(f"총 환자 수: {len(patients_new)}명")
print(patients_new[0])

총 환자 수: 9명
{'patient_id': 'Q001', 'age': 33, 'sex': 'F', 'ldl_cholesterol': 95, 'smoking': 0, 'family_history': 0}


---
# 1️⃣ 범위 (Range)

## 💡 이론: 범위란?
범위는 데이터에서 **가장 큰 값과 가장 작은 값의 차이**입니다. 반에서 "키가 제일 큰 학생과 제일 작은 학생의 키 차이"를 구하는 것과 같습니다.

$$
범위 = 최댓값 - 최솟값
$$

범위는 계산이 아주 간단하지만, **극단값 두 개(최댓값, 최솟값)에만 의존**하기 때문에 전체 데이터의 흩어진 정도를 정확히 보여주지는 못합니다. (그래서 분산·표준편차·IQR을 함께 살펴보는 것입니다.)

### 📝 문제 1
LDL 콜레스테롤 값의 **범위**를 계산하세요.


In [19]:
# TODO
ldl_values = [p["ldl_cholesterol"] for p in patients_new]


max_ldl = max(ldl_values)
min_ldl = min(ldl_values)

ldl_range =  max_ldl - min_ldl# TODO: 최댓값 - 최솟값

print("LDL 범위:", ldl_range)

LDL 범위: 157


<details>
<summary>✅ 정답 확인</summary>

```python
ldl_values = [p["ldl_cholesterol"] for p in patients_new]
ldl_range = max(ldl_values) - min(ldl_values)
print("LDL 범위:", ldl_range)   # 245 - 88 = 157
```

**해석**: LDL 범위는 157mg/dL로 매우 큽니다. 하지만 이 값은 오직 Q009(245)와 Q004(88) 두 값에만 의존하므로, "나머지 7명의 데이터가 실제로 얼마나 퍼져 있는지"는 알려주지 못합니다. 범위만으로 데이터를 판단하면 안 되는 이유입니다.
</details>


---
# 2️⃣ 분산과 표준편차

## 💡 이론 복습
표준편차는 "값들이 평균에서 얼마나 떨어져 있는가"를 나타냅니다. 양궁 선수가 화살을 쐈을 때, 평균 점수는 같아도 화살이 과녁 중심에 촘촘히 모여 있는지(표준편차 작음), 여기저기 흩어져 있는지(표준편차 큼)를 구분해주는 값입니다.

$$
분산 = \frac{\sum(x_i-\bar{x})^2}{n} \qquad 표준편차 = \sqrt{분산}
$$

### 📝 문제 2
**흡연자(smoking=1) 그룹**과 **비흡연자(smoking=0) 그룹**의 LDL 콜레스테롤 **평균과 표준편차**를 각각 구하고, 어느 그룹의 LDL 수치가 더 들쭉날쭉한지 비교해보세요.


In [20]:
import statistics as stats

smokers_ldl = [p["ldl_cholesterol"] for p in patients_new if p["smoking"] == 1]      # smoking == 1인 환자들의 ldl_cholesterol 리스트
non_smokers_ldl = [p["ldl_cholesterol"] for p in patients_new if p["smoking"] == 0]  # smoking == 0인 환자들의 ldl_cholesterol 리스트

smokers_mean = stats.mean(smokers_ldl)
smokers_std = round(stats.pstdev(smokers_ldl),2)
non_smokers_mean = stats.mean(non_smokers_ldl)
non_smokers_std = round(stats.pstdev(non_smokers_ldl),2)

print("흡연자 평균/표준편차:", smokers_mean, smokers_std)
print("비흡연자 평균/표준편차:", non_smokers_mean, non_smokers_std)

흡연자 평균/표준편차: 150 55.34
비흡연자 평균/표준편차: 97.8 5.91


<details>
<summary>✅ 정답 확인</summary>

```python
smokers_ldl = [p["ldl_cholesterol"] for p in patients_new if p["smoking"] == 1]
non_smokers_ldl = [p["ldl_cholesterol"] for p in patients_new if p["smoking"] == 0]

smokers_mean = stats.mean(smokers_ldl)
smokers_std = stats.pstdev(smokers_ldl)
non_smokers_mean = stats.mean(non_smokers_ldl)
non_smokers_std = stats.pstdev(non_smokers_ldl)

print("흡연자 평균/표준편차:", round(smokers_mean, 1), round(smokers_std, 1))       # 150.0 / 55.3
print("비흡연자 평균/표준편차:", round(non_smokers_mean, 1), round(non_smokers_std, 1))  # 97.8 / 5.9
```

**해석**: 흡연자 그룹은 평균(150.0)도 비흡연자(97.8)보다 훨씬 높고, 표준편차(55.3)도 비흡연자(5.9)보다 훨씬 큽니다. 즉 흡연자 그룹은 평균 LDL 자체가 높을 뿐 아니라, 사람마다 수치 편차도 매우 큽니다. 다만 흡연자 표준편차가 큰 이유는 Q009(245)라는 극단값 하나가 4명뿐인 작은 그룹에 크게 영향을 주었기 때문이라는 점도 함께 고려해야 합니다.
</details>


---
# 3️⃣ 사분위수와 IQR

## 💡 이론 복습
마라톤 완주자를 순위대로 세운 뒤 4등분한 지점이 Q1, Q2(중앙값), Q3입니다. IQR(Q3-Q1)은 "중간 50%가 얼마나 넓게 퍼져 있는가"를 보여주며, 극단값의 영향을 적게 받는다는 장점이 있습니다.

### 📝 문제 3
전체 9명의 LDL 콜레스테롤에 대해 **Q1, Q2, Q3, IQR**을 구하고, **이상치 후보**를 찾아보세요.


In [21]:
# TODO
def calculate_quartiles(values):
    sorted_values = sorted(values)
    n = len(sorted_values)
    middle = n // 2
    if n % 2 == 0:
        lower_half, upper_half = sorted_values[:middle], sorted_values[middle:]
    else:
        lower_half, upper_half = sorted_values[:middle], sorted_values[middle + 1:]
    return stats.median(lower_half), stats.median(sorted_values), stats.median(upper_half)

q1, q2, q3 = calculate_quartiles(ldl_values)  # TODO: calculate_quartiles(ldl_values) 사용
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
ldl_outliers = [value for value in ldl_values if value < lower_bound or value > upper_bound]

print("Q1, Q2, Q3:", q1, q2, q3)
print("IQR:", iqr)
print("이상치 후보:", ldl_outliers)

Q1, Q2, Q3: 97.0 105 122.5
IQR: 25.5
이상치 후보: [245]


<details>
<summary>✅ 정답 확인</summary>

```python
q1, q2, q3 = calculate_quartiles(ldl_values)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
ldl_outliers = [v for v in ldl_values if v < lower_bound or v > upper_bound]

print("Q1, Q2, Q3:", q1, q2, q3)     # 97.0, 105, 122.5
print("IQR:", iqr)                    # 25.5
print("정상 범위:", lower_bound, "~", upper_bound)   # 58.75 ~ 160.75
print("이상치 후보:", ldl_outliers)   # [245]
```

**해석**: IQR 기준으로 정상 범위는 약 58.75~160.75이며, Q009의 245mg/dL만 이상치 후보로 확인됩니다. 범위(문제 1)에서는 157이라는 큰 수치만 보였지만, IQR로 보면 "중간 50% 환자들은 사실 25.5mg/dL 정도의 비교적 좁은 범위 안에 몰려 있다"는 훨씬 정확한 그림을 알 수 있습니다.
</details>


---
# 4️⃣ 1일차 종합 미니 리포트

지금까지 계산한 내용을 종합해서 하나의 요약 리포트를 완성해봅니다.

### 📝 문제 4
아래 딕셔너리를 완성하세요.


In [22]:
# TODO

report = {
    "patient_count": len(patients_new),        # 총 환자 수
    "mean_age": round(stats.mean(p["age"] for p in patients_new), 0),             # 평균 나이
    "mean_ldl": round(stats.mean(ldl_values),1),             # 평균 LDL
    "std_ldl": round(stats.pstdev(ldl_values),1),              # LDL 표준편차 (모집단)
    "iqr_ldl": iqr,              # LDL IQR (문제 3에서 구한 값 재사용)
    "smoking_ratio": round(sum(p["smoking"]for p in patients_new)/len(patients_new) * 100, 1),        # 흡연자 비율(%)
    "family_history_ratio": round(sum(p["family_history"]for p in patients_new)/len(patients_new) * 100, 1), # 가족력 있는 환자 비율(%)
}

print(report)

{'patient_count': 9, 'mean_age': 49.0, 'mean_ldl': 121, 'std_ldl': 45.3, 'iqr_ldl': 25.5, 'smoking_ratio': 44.4, 'family_history_ratio': 55.6}


<details>
<summary>✅ 정답 확인</summary>

```python
report = {
    "patient_count": len(patients_new),
    "mean_age": round(stats.mean([p["age"] for p in patients_new]), 1),
    "mean_ldl": round(stats.mean(ldl_values), 1),
    "std_ldl": round(stats.pstdev(ldl_values), 1),
    "iqr_ldl": iqr,
    "smoking_ratio": round(sum(1 for p in patients_new if p["smoking"] == 1) / len(patients_new) * 100, 1),
    "family_history_ratio": round(sum(1 for p in patients_new if p["family_history"] == 1) / len(patients_new) * 100, 1),
}
print(report)
```
</details>


---
# 5️⃣ 분포의 이해 (2일차 시작)

## 💡 이론 복습
데이터가 평균을 중심으로 좌우 고르게 퍼져 있으면 **대칭분포**, 대부분 낮은 값인데 소수만 매우 높으면 **오른쪽 치우침**, 반대면 **왼쪽 치우침**입니다. 오른쪽으로 치우친 데이터는 평균이 중앙값보다 큽니다.

### 📝 문제 5
LDL 콜레스테롤 데이터의 **평균과 중앙값**을 비교해서, 이 데이터가 대칭에 가까운지 / 오른쪽으로 치우쳤는지 / 왼쪽으로 치우쳤는지 판단하고 이유를 설명하세요.


In [28]:
# TODO
ldl_mean = stats.mean(ldl_values)    # stats.mean(ldl_values)
ldl_median = stats.median(ldl_values) # stats.median(ldl_values)

print("평균:", ldl_mean, " 중앙값:", ldl_median)

'''
이 데이터는 중앙값보다 평균이 16 높은 오른쪽 치우침 데이터입니다.
평균이 중앙값보다 클 경우, 데이터가 왼쪽으로 몰려있고 꼬리가 오른쪽으로 나와있는 분포를 보입니다.
값이 큰 극단값인 Q009로 인해 평균이 높아졌을 것으로 예상합니다.
'''

평균: 121  중앙값: 105


<details>
<summary>✅ 정답 확인</summary>

```python
ldl_mean = stats.mean(ldl_values)     # 121.0
ldl_median = stats.median(ldl_values) # 105

print("평균:", ldl_mean, " 중앙값:", ldl_median)
```

**해석**: 평균(121.0)이 중앙값(105)보다 뚜렷하게 큽니다. 즉 이 데이터는 **오른쪽으로 치우친 분포**입니다. Q009(245)라는 극단적으로 높은 값 하나가 소수의 "긴 꼬리"를 만들어 평균을 끌어올렸기 때문입니다 (대부분 환자는 88~130 사이에 몰려 있습니다).
</details>


---
# 6️⃣ 빈도와 상대빈도

## 💡 이론 복습
빈도는 "몇 명인가"(개수), 상대빈도는 "전체 중 몇 %인가"(비율)입니다. 서로 다른 크기의 그룹을 비교할 때는 빈도가 아니라 상대빈도로 비교해야 합니다.

### 📝 문제 6
**흡연 여부**와 **가족력 여부**에 대해 각각 빈도표(빈도, 상대빈도, 백분율)를 작성하세요.


In [49]:
# TODO
from collections import Counter

for column in ["smoking","family_history"]:
  values = [p[column]for p in patients_new]
  counts = Counter(values)
  print(f"<{column}>")
  for category, count in sorted(counts.items()):
      ratio = count / len(patients_new)
      if category == 1:
        print(f"1(Yes) 빈도: {count}/ 상대 빈도: {ratio:.1f}/ 백분율: {ratio * 100:.1f}%")
      elif category == 0:
        print(f"0(No)  빈도: {count}/ 상대 빈도: {ratio:.1f}/ 백분율: {ratio * 100:.1f}%")



# 각각 반복문으로 빈도, 상대빈도, 백분율 출력

<smoking>
0(No)  빈도: 5/ 상대 빈도: 0.6/ 백분율: 55.6%
1(Yes) 빈도: 4/ 상대 빈도: 0.4/ 백분율: 44.4%
<family_history>
0(No)  빈도: 4/ 상대 빈도: 0.4/ 백분율: 44.4%
1(Yes) 빈도: 5/ 상대 빈도: 0.6/ 백분율: 55.6%


<details>
<summary>✅ 정답 확인</summary>

```python
from collections import Counter

for column in ["smoking", "family_history"]:
    values = [p[column] for p in patients_new]
    counts = Counter(values)
    print(f"--- {column} ---")
    for category, count in sorted(counts.items()):
        ratio = count / len(patients_new)
        print(category, count, f"{ratio * 100:.1f}%")
```

**결과**: 흡연자 4명(44.4%), 비흡연자 5명(55.6%) / 가족력 있음 5명(55.6%), 없음 4명(44.4%)
</details>


---
# 7️⃣  조건부 비율과 조건부 확률

## 💡 이론 복습
"~중에서"라는 말 앞에 오는 대상이 **분모(기준 집단)**입니다.

$$
P(A|B) = \frac{A와\ B를\ 모두\ 만족하는\ 수}{B를\ 만족하는\ 전체\ 수}
$$

### 📝 문제 7
다음 두 값을 각각 계산하고, 두 값이 왜 다른지 설명하세요.

1. **가족력이 있는 환자 중** 흡연자의 비율
2. **흡연자 중** 가족력이 있는 환자의 비율


In [51]:
# TODO
family_history_patients = [p for p in patients_new if p['family_history'] == 1]      # family_history == 1인 환자 리스트
family_history_smokers = [p for p in family_history_patients if p['smoking'] == 1]        # 그 중 smoking == 1인 환자 리스트
ratio_1 = len(family_history_smokers) / len(family_history_patients) *100                       # 1번 질문의 답

smokers = [p for p in patients_new if p['smoking'] == 1]                        # smoking == 1인 환자 리스트
smokers_with_family_history = [p for p in smokers if p['family_history'] == 1]     # 그 중 family_history == 1인 환자 리스트
ratio_2 = len(smokers_with_family_history) / len(smokers) *100                       # 2번 질문의 답

print("가족력 있는 환자 중 흡연 비율:", ratio_1)
print("흡연자 중 가족력 있는 비율:", ratio_2)

가족력 있는 환자 중 흡연 비율: 80.0
흡연자 중 가족력 있는 비율: 100.0


<details>
<summary>✅ 정답 확인</summary>

```python
family_history_patients = [p for p in patients_new if p["family_history"] == 1]
family_history_smokers = [p for p in family_history_patients if p["smoking"] == 1]
ratio_1 = len(family_history_smokers) / len(family_history_patients)
print(f"가족력 있는 환자 중 흡연 비율: {ratio_1 * 100:.1f}%")   # 4/5 = 80.0%

smokers = [p for p in patients_new if p["smoking"] == 1]
smokers_with_family_history = [p for p in smokers if p["family_history"] == 1]
ratio_2 = len(smokers_with_family_history) / len(smokers)
print(f"흡연자 중 가족력 있는 비율: {ratio_2 * 100:.1f}%")      # 4/4 = 100.0%
```

**해석**: 두 값이 다릅니다(80.0% vs 100.0%) — **분모(기준 집단)가 다르기 때문**입니다. 1번은 "가족력이 있는 환자 5명"을 기준으로 계산했고, 2번은 "흡연자 4명"을 기준으로 계산했습니다. "~중에서"라는 표현이 가리키는 대상이 항상 분모라는 점을 다시 한번 확인할 수 있습니다.
</details>
